In [0]:
%python
# 08_token_costs: measure how many tokens each model/prompt uses per bug,
# so we can turn it into "dollars per 1,000 bugs".
#
# Why Python: ai_query() only hands back the answer, not the token counts.
# Calling the same endpoints from Python returns a "usage" section with
# prompt_tokens (what we sent) and completion_tokens (what the model wrote,
# including its hidden thinking). We only need a sample of bugs to get a
# good average, so this uses 50 bugs per configuration.
 
# ===== Cell 1: settings =====
CONFIGS = [  # (model endpoint, prompt version)
    ("databricks-gpt-oss-20b", "v2"),
    ("databricks-gpt-oss-120b", "v2"),
    ("databricks-llama-4-maverick", "v2"),
]
SAMPLE_SIZE = 50
QUESTION = "\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else."

In [0]:
%python
# ===== Cell 2: call the models and record tokens =====
from mlflow.deployments import get_deploy_client
from pyspark.sql import Row
 
client = get_deploy_client("databricks")  # uses the notebook's own login, no key needed
prompts = {r.version: r.team_guide for r in spark.table("support.bugzilla.prompts").collect()}
bugs = spark.table("support.bugzilla.eval_set_2").orderBy("id").limit(SAMPLE_SIZE).collect()
 
rows = []
for model, version in CONFIGS:
    for b in bugs:
        content = (prompts[version] + "\n\nBug title: " + b.summary
                   + "\nBug description: " + (b.description or "")[:2000] + QUESTION)
        resp = client.predict(endpoint=model, inputs={
            "messages": [{"role": "user", "content": content}],
            "max_tokens": 2000, "temperature": 0.0})
        u = resp["usage"]
        rows.append(Row(model=model, prompt_version=version, id=b.id,
                        prompt_tokens=u["prompt_tokens"], completion_tokens=u["completion_tokens"]))
    print("done", model, version)
 
spark.createDataFrame(rows).write.mode("append").saveAsTable("support.bugzilla.token_usage")


In [0]:
%python
# ===== Cell 3: prices (list price, US dollars per 1M tokens) =====
# From databricks.com/product/pricing/foundation-model-serving, at the example rate of $0.07 per DBU.
# Free Edition does not charge you; these are what a paying customer would pay.
prices = spark.createDataFrame([
    ("databricks-gpt-oss-20b",      0.07, 0.30),
    ("databricks-gpt-oss-120b",     0.15, 0.60),
    ("databricks-llama-4-maverick", 0.50, 1.50),
], "model STRING, input_per_m DOUBLE, output_per_m DOUBLE")
prices.write.mode("overwrite").saveAsTable("support.bugzilla.model_prices")

In [0]:
SELECT * FROM support.bugzilla.token_usage tu JOIN support.bugzilla.model_prices mp ON tu.model = mp.model

In [0]:
-- # ===== Cell 4: cost per 1,000 bugs =====
-- # Yours to write, in SQL (%sql cell): join token_usage to model_prices on model,
-- # take the average prompt and completion tokens per bug, then
-- #   cost per bug = avg_prompt / 1e6 * input_per_m + avg_completion / 1e6 * output_per_m
-- # and multiply by 1,000.
SELECT
  tu.model,
  prompt_version,
  ROUND(
    1000 * (AVG(prompt_tokens) / 1e6 * MAX(input_per_m) + AVG(completion_tokens) / 1e6 * MAX(output_per_m)),
    4
  ) AS cost_per_1k_bugs
FROM
  support.bugzilla.token_usage tu
    JOIN support.bugzilla.model_prices mp
      ON tu.model = mp.model
GROUP BY
  tu.model,
  prompt_version